# 第13回: ACT - Action Chunking with Transformers

## 概要

ACT (Action Chunking with Transformers) は、ロボット操作のための模倣学習手法です。
主な特徴は以下の通りです:

- **Conditional VAE (CVAE)**: 行動の多様性を潜在変数でモデル化
- **Transformer Encoder/Decoder**: 観測と潜在変数から行動チャンクを生成
- **Action Chunking**: 複数ステップの行動をまとめて予測し、compounding errorを軽減
- **Temporal Ensemble**: 重複するチャンクを加重平均して滑らかな行動を実現

この回では、ACTの各コンポーネントを段階的に実装し、理解を深めます。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. Conditional VAE (CVAE)

### VAEの基本概念

Variational Autoencoder (VAE) は、データの潜在表現を学習する生成モデルです。
ACTでは **Conditional VAE** を使い、観測 $o$ を条件として行動列 $a_{1:H}$ をモデル化します。

**学習時**: エンコーダが $(o, a_{1:H})$ から潜在変数 $z$ の分布 $q(z|o,a)$ を推定\
**推論時**: $z \sim \mathcal{N}(0, I)$ からサンプルし、デコーダが行動を生成

### KL Divergence

CVAEの損失関数は以下の2項からなります:

$$\mathcal{L} = \underbrace{\|a - \hat{a}\|^2}_{\text{再構成損失}} + \beta \cdot \underbrace{D_{\mathrm{KL}}(q(z|o,a) \| p(z))}_{\text{KL正則化}}$$

KLダイバージェンスは事後分布 $q(z|o,a)$ を事前分布 $p(z)=\mathcal{N}(0,I)$ に近づけます。

### Reparameterization Trick

$z$ のサンプリングを微分可能にするため、以下の変換を用います:

$$z = \mu + \epsilon \cdot \sigma, \quad \epsilon \sim \mathcal{N}(0, I)$$

In [ ]:
class CVAE(nn.Module):
    """Conditional VAE: 観測と行動列から潜在変数zを推定"""

    def __init__(self, action_dim, obs_dim, latent_dim, horizon):
        super().__init__()
        self.latent_dim = latent_dim
        input_dim = obs_dim + action_dim * horizon
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

    def encode(self, obs, actions):
        """観測と行動列からmu, logvarを推定"""
        flat_actions = actions.reshape(actions.shape[0], -1)
        h = self.encoder(torch.cat([obs, flat_actions], dim=-1))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        """Reparameterization trick"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def kl_divergence(self, mu, logvar):
        """KL(q(z|o,a) || N(0,I))"""
        return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=-1).mean()


# テスト
obs_dim, action_dim, latent_dim, horizon = 32, 7, 16, 10
cvae = CVAE(action_dim, obs_dim, latent_dim, horizon)

batch = 4
obs = torch.randn(batch, obs_dim)
actions = torch.randn(batch, horizon, action_dim)
mu, logvar = cvae.encode(obs, actions)
z = cvae.reparameterize(mu, logvar)
kl = cvae.kl_divergence(mu, logvar)

print(f"mu shape:     {mu.shape}")       # [4, 16]
print(f"logvar shape: {logvar.shape}")   # [4, 16]
print(f"z shape:      {z.shape}")         # [4, 16]
print(f"KL divergence: {kl.item():.4f}")

## 2. Transformer Encoder

ACTのTransformer Encoderは、観測トークンと潜在変数 $z$ を入力として受け取り、
デコーダのためのコンテキスト表現を生成します。

### Positional Encoding

Transformerは入力の順序情報を持たないため、Positional Encodingを追加します。
正弦波ベースのPEを使用します:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

### エンコーダの入力

- 観測 $o$ を線形変換してトークン化
- 潜在変数 $z$ もトークンとして追加
- `[z_token, obs_token]` の系列をTransformer Encoderに入力

In [ ]:
class PositionalEncoding(nn.Module):
    """正弦波Positional Encoding"""

    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class ACTEncoder(nn.Module):
    """観測+潜在変数をエンコードするTransformer Encoder"""

    def __init__(self, obs_dim, latent_dim, d_model=128, nhead=4, num_layers=2):
        super().__init__()
        self.obs_proj = nn.Linear(obs_dim, d_model)
        self.z_proj = nn.Linear(latent_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            batch_first=True, dropout=0.1,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, obs, z):
        """
        obs: (B, obs_dim) -> obs token
        z:   (B, latent_dim) -> z token
        戻り値: (B, 2, d_model) のエンコーダ出力
        """
        obs_token = self.obs_proj(obs).unsqueeze(1)  # (B, 1, d_model)
        z_token = self.z_proj(z).unsqueeze(1)        # (B, 1, d_model)
        tokens = torch.cat([z_token, obs_token], dim=1)  # (B, 2, d_model)
        tokens = self.pos_enc(tokens)
        return self.transformer_encoder(tokens)


# テスト
d_model = 128
encoder = ACTEncoder(obs_dim, latent_dim, d_model=d_model)
z = torch.randn(batch, latent_dim)
enc_out = encoder(obs, z)
print(f"Encoder output shape: {enc_out.shape}")  # [4, 2, 128]

## 3. Transformer Decoder

Transformer Decoderは **learnable action queries** を使って、エンコーダ出力から
行動チャンクを生成します。

### Learnable Action Queries

- 各タイムステップに対応する学習可能なクエリベクトルを用意
- horizon 個のクエリがCross-Attentionでエンコーダ出力を参照
- 各クエリの出力を線形変換して行動を予測

### Cross-Attention

デコーダの各クエリは、エンコーダ出力（観測+潜在変数の表現）に対して
Cross-Attentionを行い、必要な情報を取得します。

In [ ]:
class ACTDecoder(nn.Module):
    """Learnable action queriesとCross-Attentionによる行動チャンク生成"""

    def __init__(self, action_dim, horizon, d_model=128, nhead=4, num_layers=2):
        super().__init__()
        self.action_queries = nn.Parameter(torch.randn(1, horizon, d_model) * 0.02)
        self.pos_enc = PositionalEncoding(d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256,
            batch_first=True, dropout=0.1,
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.action_head = nn.Linear(d_model, action_dim)

    def forward(self, memory):
        """
        memory: (B, seq_len, d_model) エンコーダ出力
        戻り値: (B, horizon, action_dim) 予測行動チャンク
        """
        B = memory.shape[0]
        queries = self.action_queries.expand(B, -1, -1)  # (B, horizon, d_model)
        queries = self.pos_enc(queries)
        decoded = self.transformer_decoder(queries, memory)  # (B, horizon, d_model)
        return self.action_head(decoded)  # (B, horizon, action_dim)


# テスト
decoder = ACTDecoder(action_dim, horizon, d_model=d_model)
actions_pred = decoder(enc_out)
print(f"Predicted actions shape: {actions_pred.shape}")  # [4, 10, 7]

## 4. Action Chunking と Temporal Ensemble

### Action Chunking

従来の1ステップ予測では、予測誤差が蓄積する **compounding error** が問題でした。
Action Chunkingでは **horizon $H$ ステップの行動をまとめて予測** することで、
この問題を軽減します。

- 1回の推論で $a_t, a_{t+1}, \ldots, a_{t+H-1}$ を同時に予測
- 各チャンク内では一貫性のある行動系列が得られる

### Temporal Ensemble

推論時には、異なるタイムステップで予測されたチャンクが重複します。
Temporal Ensembleでは、重複する予測を **指数減衰の重み** で加重平均します:

$$a_t = \frac{\sum_{k} w_k \cdot \hat{a}_t^{(k)}}{\sum_{k} w_k}, \quad w_k = \exp(-m \cdot k)$$

ここで $m$ は減衰係数、$k$ は予測が何ステップ前に行われたかを表します。
新しい予測ほど重みが大きくなります。

In [ ]:
def temporal_ensemble(chunks, decay=0.1):
    """
    Temporal Ensemble: 重複するチャンクを加重平均

    Args:
        chunks: list of (start_t, action_chunk) タプル
                action_chunk: (horizon, action_dim)
        decay: 指数減衰の係数

    Returns:
        ensembled: (T, action_dim) 合成された行動列
    """
    # 全体の時間長を計算
    max_t = max(s + c.shape[0] for s, c in chunks)
    action_dim = chunks[0][1].shape[-1]
    weighted_sum = torch.zeros(max_t, action_dim)
    weight_sum = torch.zeros(max_t, 1)

    for start_t, chunk in chunks:
        H = chunk.shape[0]
        for j in range(H):
            t = start_t + j
            w = np.exp(-decay * j)  # 新しいステップほど重みが大きい
            weighted_sum[t] += w * chunk[j]
            weight_sum[t] += w

    mask = weight_sum.squeeze() > 0
    ensembled = torch.zeros_like(weighted_sum)
    ensembled[mask] = weighted_sum[mask] / weight_sum[mask]
    return ensembled


# Temporal Ensembleの効果を可視化
horizon_vis = 5
num_steps = 15
action_dim_vis = 1

# ダミー行動チャンクを生成（ノイズ付き正弦波）
t_all = torch.linspace(0, 3 * np.pi, num_steps + horizon_vis)
chunks_list = []
for t in range(0, num_steps, 2):  # 2ステップおきに予測
    gt = torch.sin(t_all[t:t+horizon_vis]).unsqueeze(-1)
    noisy = gt + 0.15 * torch.randn_like(gt)
    chunks_list.append((t, noisy))

ensembled = temporal_ensemble(chunks_list, decay=0.1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 左: 各チャンクを色分け表示
ax = axes[0]
colors = plt.cm.tab10(np.linspace(0, 1, len(chunks_list)))
for i, (s, c) in enumerate(chunks_list):
    ts = np.arange(s, s + c.shape[0])
    ax.plot(ts, c[:, 0].numpy(), "o-", color=colors[i], alpha=0.5, markersize=3,
            label=f"chunk t={s}" if i < 5 else None)
ax.plot(torch.sin(t_all[:num_steps+horizon_vis]).numpy(), "k--", alpha=0.5, label="Ground Truth")
ax.set_title("各チャンクの予測")
ax.set_xlabel("Time step")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 右: Temporal Ensemble結果
ax = axes[1]
T = ensembled.shape[0]
ax.plot(range(T), ensembled[:, 0].numpy(), "b-", linewidth=2, label="Temporal Ensemble")
ax.plot(torch.sin(t_all[:T]).numpy(), "k--", alpha=0.5, label="Ground Truth")
ax.set_title("Temporal Ensemble後")
ax.set_xlabel("Time step")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. ACT全体モデル

各コンポーネントを統合して、ACTモデル全体を構築します。

**学習時のフロー**:
1. CVAE Encoderが $(o, a_{1:H})$ から $\mu, \log\sigma^2$ を推定
2. Reparameterization trickで $z$ をサンプル
3. Transformer Encoderが $[z, o]$ をエンコード
4. Transformer Decoderが action queries からチャンクを生成
5. 再構成損失 + KL損失で学習

**推論時のフロー**:
1. $z \sim \mathcal{N}(0, I)$ をサンプル
2. Transformer Encoder → Decoder で行動チャンクを生成

In [ ]:
class ACT(nn.Module):
    """Action Chunking with Transformers"""

    def __init__(self, action_dim, obs_dim, latent_dim=16, horizon=10,
                 d_model=128, nhead=4, enc_layers=2, dec_layers=2):
        super().__init__()
        self.latent_dim = latent_dim
        self.horizon = horizon

        # CVAE Encoder (学習時のみ使用)
        self.cvae = CVAE(action_dim, obs_dim, latent_dim, horizon)

        # Transformer Encoder
        self.encoder = ACTEncoder(obs_dim, latent_dim, d_model, nhead, enc_layers)

        # Transformer Decoder
        self.decoder = ACTDecoder(action_dim, horizon, d_model, nhead, dec_layers)

    def forward(self, obs, actions=None):
        """
        学習時: obs, actions -> predicted_actions, mu, logvar
        推論時: obs -> predicted_actions
        """
        B = obs.shape[0]

        if actions is not None:
            # 学習時: CVAEでzをエンコード
            mu, logvar = self.cvae.encode(obs, actions)
            z = self.cvae.reparameterize(mu, logvar)
        else:
            # 推論時: 標準正規分布からサンプル
            mu, logvar = None, None
            z = torch.randn(B, self.latent_dim, device=obs.device)

        # Transformer Encoder -> Decoder
        memory = self.encoder(obs, z)
        pred_actions = self.decoder(memory)

        return pred_actions, mu, logvar

    def compute_loss(self, obs, actions, beta=1.0):
        """学習用の損失計算"""
        pred_actions, mu, logvar = self.forward(obs, actions)
        recon_loss = F.mse_loss(pred_actions, actions)
        kl_loss = self.cvae.kl_divergence(mu, logvar)
        return recon_loss + beta * kl_loss, recon_loss, kl_loss


# モデル構築テスト
model = ACT(action_dim=7, obs_dim=32, latent_dim=16, horizon=10)
obs_test = torch.randn(4, 32)
act_test = torch.randn(4, 10, 7)

# 学習モード
pred, mu, logvar = model(obs_test, act_test)
print(f"学習時 - 予測行動: {pred.shape}, mu: {mu.shape}, logvar: {logvar.shape}")

# 推論モード
with torch.no_grad():
    pred_inf, _, _ = model(obs_test)
    print(f"推論時 - 予測行動: {pred_inf.shape}")

total_params = sum(p.numel() for p in model.parameters())
print(f"総パラメータ数: {total_params:,}")

## 6. 学習と可視化

Robomimicスタイルのダミーデータを生成し、ACTモデルを学習します。
損失曲線と予測チャンクの可視化を行います。

In [ ]:
# ダミーデータ生成 (Robomimicスタイル: リーチング軌道)
def generate_dummy_data(num_episodes=200, horizon=10, obs_dim=32, action_dim=7):
    """正弦波ベースのダミーロボット軌道を生成"""
    obs_list, act_list = [], []
    for i in range(num_episodes):
        # ランダムな周波数と振幅
        freq = 0.5 + np.random.rand() * 1.5
        amp = 0.3 + np.random.rand() * 0.7
        phase = np.random.rand() * 2 * np.pi
        t = np.linspace(0, 2 * np.pi * freq, horizon)
        # 各関節の行動
        actions = np.zeros((horizon, action_dim))
        for j in range(action_dim):
            actions[:, j] = amp * np.sin(t + phase + j * 0.5) + 0.05 * np.random.randn(horizon)
        # 観測: 行動の初期状態 + ランダム特徴
        obs = np.zeros(obs_dim)
        obs[:action_dim] = actions[0]
        obs[action_dim:] = np.random.randn(obs_dim - action_dim) * 0.1
        obs_list.append(obs)
        act_list.append(actions)
    return torch.tensor(np.array(obs_list), dtype=torch.float32), \
           torch.tensor(np.array(act_list), dtype=torch.float32)


obs_data, act_data = generate_dummy_data(num_episodes=500, horizon=10, obs_dim=32, action_dim=7)
dataset = TensorDataset(obs_data, act_data)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)
print(f"データセット: 観測 {obs_data.shape}, 行動 {act_data.shape}")

In [ ]:
# 学習ループ
model = ACT(action_dim=7, obs_dim=32, latent_dim=16, horizon=10,
            d_model=128, nhead=4, enc_layers=2, dec_layers=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_epochs = 80
history = {"total": [], "recon": [], "kl": []}

# KLアニーリング: betaを徐々に増加
for epoch in range(num_epochs):
    model.train()
    beta = min(1.0, epoch / 30)  # 30エポックで1.0に到達
    epoch_losses = {"total": 0, "recon": 0, "kl": 0}
    n_batches = 0

    for obs_b, act_b in dataloader:
        obs_b, act_b = obs_b.to(device), act_b.to(device)
        total_loss, recon_loss, kl_loss = model.compute_loss(obs_b, act_b, beta=beta)
        optimizer.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_losses["total"] += total_loss.item()
        epoch_losses["recon"] += recon_loss.item()
        epoch_losses["kl"] += kl_loss.item()
        n_batches += 1

    for k in epoch_losses:
        history[k].append(epoch_losses[k] / n_batches)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Total: {history['total'][-1]:.4f} | "
              f"Recon: {history['recon'][-1]:.4f} | KL: {history['kl'][-1]:.4f} | beta: {beta:.2f}")

In [ ]:
# 損失曲線の可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, key, title in zip(axes, ["total", "recon", "kl"],
                           ["Total Loss", "Reconstruction Loss", "KL Divergence"]):
    ax.plot(history[key], linewidth=1.5)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 予測チャンクの可視化
model.eval()
num_vis = 4
indices = np.random.choice(len(obs_data), num_vis, replace=False)

fig, axes = plt.subplots(num_vis, 2, figsize=(14, 3 * num_vis))

for row, idx in enumerate(indices):
    obs_i = obs_data[idx:idx+1].to(device)
    act_gt = act_data[idx].numpy()

    # 学習時（CVAEエンコード使用）
    with torch.no_grad():
        pred_train, _, _ = model(obs_i, act_data[idx:idx+1].to(device))
        pred_train = pred_train.cpu().numpy()[0]
        # 推論時（z ~ N(0,I)）
        pred_infer, _, _ = model(obs_i)
        pred_infer = pred_infer.cpu().numpy()[0]

    # 学習時の予測
    ax = axes[row, 0]
    for j in range(min(3, act_gt.shape[-1])):
        ax.plot(act_gt[:, j], "k--", alpha=0.5, label=f"GT dim{j}" if row == 0 else None)
        ax.plot(pred_train[:, j], "-", alpha=0.8, label=f"Pred dim{j}" if row == 0 else None)
    ax.set_title(f"Sample {idx} - 学習時 (CVAEエンコード)")
    ax.grid(True, alpha=0.3)
    if row == 0:
        ax.legend(fontsize=7)

    # 推論時の予測
    ax = axes[row, 1]
    for j in range(min(3, act_gt.shape[-1])):
        ax.plot(act_gt[:, j], "k--", alpha=0.5)
        ax.plot(pred_infer[:, j], "-", alpha=0.8)
    ax.set_title(f"Sample {idx} - 推論時 (z~N(0,I))")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 演習問題

### 演習1: CVAEのencode/reparameterizeを実装

以下のクラスの `encode` と `reparameterize` メソッドを実装してください。

- `encode`: 観測と行動列を結合してエンコーダに通し、`mu` と `logvar` を返す
- `reparameterize`: Reparameterization trickで潜在変数 $z$ をサンプル

In [ ]:
class MyCVAE(nn.Module):
    def __init__(self, action_dim, obs_dim, latent_dim, horizon):
        super().__init__()
        self.latent_dim = latent_dim
        input_dim = obs_dim + action_dim * horizon
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)

    def encode(self, obs, actions):
        # ここにコードを書いてください
        pass

    def reparameterize(self, mu, logvar):
        # ここにコードを書いてください
        pass

# テスト
my_cvae = MyCVAE(action_dim=7, obs_dim=32, latent_dim=16, horizon=10)
test_obs = torch.randn(4, 32)
test_actions = torch.randn(4, 10, 7)
mu, logvar = my_cvae.encode(test_obs, test_actions)
z = my_cvae.reparameterize(mu, logvar)
assert mu.shape == (4, 16), f"mu shape should be (4, 16), got {mu.shape}"
assert z.shape == (4, 16), f"z shape should be (4, 16), got {z.shape}"
print("テスト通過!")

<details>
<summary><b>回答例を表示</b></summary>

```python
def encode(self, obs, actions):
    flat_actions = actions.reshape(actions.shape[0], -1)
    h = self.encoder(torch.cat([obs, flat_actions], dim=-1))
    return self.fc_mu(h), self.fc_logvar(h)

def reparameterize(self, mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

```

</details>

### 演習2: ACT全体を組み立て、ダミーデータで学習・推論

上で定義したコンポーネント（CVAE, ACTEncoder, ACTDecoder）を使って
ACTモデルを構築し、ダミーデータで10エポック学習してください。
学習後に推論モード（actionsなし）で行動チャンクを生成してください。

In [ ]:
# ダミーデータ
train_obs, train_act = generate_dummy_data(num_episodes=100, horizon=10, obs_dim=32, action_dim=7)

# ここにコードを書いてください
# 1. ACTモデルを構築
# 2. 10エポック学習
# 3. 推論モードで行動チャンクを生成

# model = ...
# for epoch in range(10):
#     ...

# 推論
# with torch.no_grad():
#     ...

<details>
<summary><b>回答例を表示</b></summary>

```python
train_obs, train_act = generate_dummy_data(num_episodes=100, horizon=10, obs_dim=32, action_dim=7)
train_dataset = TensorDataset(train_obs, train_act)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

my_model = ACT(action_dim=7, obs_dim=32, latent_dim=16, horizon=10)
my_optimizer = torch.optim.AdamW(my_model.parameters(), lr=1e-3)

for epoch in range(10):
    my_model.train()
    total = 0
    n = 0
    for ob, ac in train_loader:
        loss, _, _ = my_model.compute_loss(ob, ac, beta=min(1.0, epoch/5))
        my_optimizer.zero_grad()
        loss.backward()
        my_optimizer.step()
        total += loss.item()
        n += 1
    print(f"Epoch {epoch+1}: loss={total/n:.4f}")

my_model.eval()
with torch.no_grad():
    test_input = train_obs[:4]
    pred, _, _ = my_model(test_input)
    print(f"推論結果: {pred.shape}")  # (4, 10, 7)

```

</details>

### 演習3: Temporal Ensembleを実装し、chunk_sizeの比較

異なる `chunk_size`（horizon）でACTモデルを学習し、
Temporal Ensembleの効果を比較してください。

1. chunk_size = 5, 10, 20 の3パターンで学習
2. 各モデルの推論結果にTemporal Ensembleを適用
3. Ground Truthとの誤差を比較・可視化

In [ ]:
def my_temporal_ensemble(chunks, decay=0.1):
    """
    chunks: list of (start_t, action_chunk) タプル
    decay: 指数減衰の係数
    戻り値: (T, action_dim) 合成された行動列
    """
    # ここにコードを書いてください
    pass


# chunk_size比較の実験
chunk_sizes = [5, 10, 20]
results = {}

for cs in chunk_sizes:
    # ここにコードを書いてください
    # 1. データ生成 (horizon=cs)
    # 2. ACTモデル構築・学習
    # 3. Temporal Ensembleで行動列を合成
    # 4. 誤差を計算
    pass

# 結果の可視化
# ここにコードを書いてください

<details>
<summary><b>回答例を表示</b></summary>

```python
def my_temporal_ensemble(chunks, decay=0.1):
    max_t = max(s + c.shape[0] for s, c in chunks)
    action_dim = chunks[0][1].shape[-1]
    weighted_sum = torch.zeros(max_t, action_dim)
    weight_sum = torch.zeros(max_t, 1)
    for start_t, chunk in chunks:
        H = chunk.shape[0]
        for j in range(H):
            t = start_t + j
            w = np.exp(-decay * j)
            weighted_sum[t] += w * chunk[j]
            weight_sum[t] += w
    mask = weight_sum.squeeze() > 0
    result = torch.zeros_like(weighted_sum)
    result[mask] = weighted_sum[mask] / weight_sum[mask]
    return result

chunk_sizes = [5, 10, 20]
results = {}

for cs in chunk_sizes:
    data_obs, data_act = generate_dummy_data(num_episodes=200, horizon=cs, obs_dim=32, action_dim=7)
    ds = TensorDataset(data_obs, data_act)
    dl = DataLoader(ds, batch_size=32, shuffle=True)

    m = ACT(action_dim=7, obs_dim=32, latent_dim=16, horizon=cs)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)

    for epoch in range(30):
        m.train()
        for ob, ac in dl:
            loss, _, _ = m.compute_loss(ob, ac, beta=min(1.0, epoch/10))
            opt.zero_grad()
            loss.backward()
            opt.step()

    m.eval()
    # 連続推論 + Temporal Ensemble
    test_obs_seq = data_obs[:20]  # 20ステップ分
    chunks_list = []
    with torch.no_grad():
        for i in range(0, 20, max(1, cs // 2)):  # 半分ずつオーバーラップ
            if i >= len(test_obs_seq):
                break
            pred, _, _ = m(test_obs_seq[i:i+1])
            chunks_list.append((i, pred[0].cpu()))

    ensembled = my_temporal_ensemble(chunks_list, decay=0.1)
    results[cs] = ensembled

fig, axes = plt.subplots(1, len(chunk_sizes), figsize=(5*len(chunk_sizes), 4))
for ax, cs in zip(axes, chunk_sizes):
    r = results[cs]
    for j in range(min(3, r.shape[-1])):
        ax.plot(r[:, j].numpy(), label=f"dim {j}")
    ax.set_title(f"chunk_size = {cs}")
    ax.set_xlabel("Time step")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

```

</details>

## まとめ

この回では、ACT (Action Chunking with Transformers) の各コンポーネントを実装しました:

1. **CVAE**: 行動の多様性を潜在空間でモデル化（Reparameterization Trick, KL正則化）
2. **Transformer Encoder**: 観測と潜在変数を統合的にエンコード（Positional Encoding）
3. **Transformer Decoder**: Learnable action queriesとCross-Attentionで行動チャンクを生成
4. **Action Chunking**: 複数ステップをまとめて予測し、compounding errorを軽減
5. **Temporal Ensemble**: 重複するチャンクの加重平均で滑らかな行動を実現

ACTは特にロボットマニピュレーションタスクにおいて高い性能を示し、
比較的少ないデモンストレーションデータからの学習が可能です。